# Point Cloud 01 -- Fundamentals: Structure, Normals, and Spatial Search

> **Geo-Instructor · Point Cloud Career Track · Notebook 1 of 3**

---

A LiDAR scanner fires millions of laser pulses and records where each one bounced back.
The result: a cloud of (x, y, z) points with no connectivity, no faces, no normals.
This notebook covers the fundamental processing pipeline every 3D vision engineer must know.

```
Runtime: ~4 min  |  No GPU  |  numpy + scipy + matplotlib only
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial import KDTree
import time
np.random.seed(42)
plt.rcParams.update({'figure.facecolor':'#F4F2F0','axes.facecolor':'#F4F2F0','font.family':'monospace'})
print('Ready.')

In [ ]:
# Synthetic scene: floor + sphere + box
def sample_plane(n, xlim, ylim, z, noise=0.02):
    x = np.random.uniform(*xlim, n)
    y = np.random.uniform(*ylim, n)
    z_arr = np.full(n, z) + np.random.randn(n)*noise
    return np.column_stack([x, y, z_arr])

def sample_sphere(n, center, radius, noise=0.03):
    phi = np.random.uniform(0, np.pi, n)
    theta = np.random.uniform(0, 2*np.pi, n)
    x = center[0] + (radius + np.random.randn(n)*noise) * np.sin(phi)*np.cos(theta)
    y = center[1] + (radius + np.random.randn(n)*noise) * np.sin(phi)*np.sin(theta)
    z = center[2] + (radius + np.random.randn(n)*noise) * np.cos(phi)
    return np.column_stack([x, y, z])

def sample_box_surface(n, center, size, noise=0.02):
    pts = []
    for axis in range(3):
        for sign in [-1, 1]:
            m = n // 6
            face = np.random.uniform(-0.5, 0.5, (m, 3)) * size
            face[:, axis] = sign * size[axis]/2
            face += center
            face += np.random.randn(m, 3) * noise
            pts.append(face)
    return np.vstack(pts)

floor  = sample_plane(1500, [-3,3], [-3,3], z=0)
sphere = sample_sphere(800, center=[0, 1.2, 1.0], radius=0.8)
box    = sample_box_surface(600, center=[-1.5, -1.0, 0.5], size=np.array([1,1,1]))

cloud = np.vstack([floor, sphere, box])
# Add a few outliers
cloud = np.vstack([cloud, np.random.uniform(-3, 3, (50, 3))])

print(f'Point cloud: {len(cloud):,} points')
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')
for pts, c, label in [(floor,'#8A857E','floor'), (sphere,'steelblue','sphere'), (box,'tomato','box')]:
    ax.scatter(pts[:,0], pts[:,1], pts[:,2], s=1, c=c, alpha=0.4, label=label)
ax.set_title('Synthetic LiDAR scene'); ax.legend(markerscale=6, fontsize=8)
plt.tight_layout(); plt.show()

---
## Part 1 -- Voxel Grid Downsampling

Raw point clouds are huge. Voxel downsampling divides 3D space into a uniform grid
and keeps one representative point per voxel (typically the centroid).
This is the first step in almost every processing pipeline.

In [ ]:
def voxel_downsample(pts, voxel_size):
    """Voxel grid downsampling: keep centroid per voxel."""
    indices = np.floor(pts / voxel_size).astype(int)
    voxel_dict = {}
    for i, idx in enumerate(map(tuple, indices)):
        voxel_dict.setdefault(idx, []).append(i)
    result = [pts[idxs].mean(axis=0) for idxs in voxel_dict.values()]
    return np.array(result)

for vs in [0.05, 0.10, 0.20, 0.40]:
    down = voxel_downsample(cloud, vs)
    print(f'voxel_size={vs:.2f}m  ->  {len(down):>5,} pts  (ratio {len(down)/len(cloud)*100:.1f}%)')

down_05 = voxel_downsample(cloud, 0.08)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), subplot_kw={'projection':'3d'})
for ax, pts, title in [(axes[0], cloud, f'Original: {len(cloud):,} pts'), (axes[1], down_05, f'Downsampled: {len(down_05):,} pts (voxel=0.08m)')]:
    ax.scatter(pts[:,0], pts[:,1], pts[:,2], s=1, c=pts[:,2], cmap='viridis', alpha=0.5)
    ax.set_title(title, fontsize=9)
plt.tight_layout(); plt.show()

---
## Part 2 -- Normal Estimation via PCA

Points have no normals by default. We estimate them by looking at local neighborhoods:
fit a plane to k nearest neighbors, the normal is the smallest eigenvector of the covariance matrix.

The normal direction is ambiguous (inward vs outward) -- handled by orientation propagation.

In [ ]:
def estimate_normals(pts, k=15):
    """PCA-based normal estimation."""
    kdt = KDTree(pts)
    normals = np.zeros_like(pts)
    for i, p in enumerate(pts):
        _, idx = kdt.query(p, k=k)
        neighbors = pts[idx]
        cov = np.cov(neighbors.T)
        eigvals, eigvecs = np.linalg.eigh(cov)
        normals[i] = eigvecs[:, 0]  # smallest eigenvalue -> normal
    return normals

down = voxel_downsample(cloud, 0.12)
t0 = time.time()
normals = estimate_normals(down, k=12)
print(f'Normal estimation: {len(down)} points in {time.time()-t0:.2f}s')

# Visualize on a subset of sphere points
sphere_mask = np.linalg.norm(down - [0,1.2,1.0], axis=1) < 1.0
sp = down[sphere_mask]; sn = normals[sphere_mask]

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(sp[:,0], sp[:,1], sp[:,2], s=4, c='steelblue', alpha=0.5)
stride = 4
ax.quiver(sp[::stride,0], sp[::stride,1], sp[::stride,2],
          sn[::stride,0], sn[::stride,1], sn[::stride,2],
          length=0.12, normalize=True, color='tomato', alpha=0.6)
ax.set_title('Surface normals on sphere region')
plt.tight_layout(); plt.show()
print('Normal consistency: check that sphere normals point roughly radially outward.')

---
## Part 3 -- Statistical Outlier Removal (SOR)

For each point, compute the mean distance to its k nearest neighbors.
Points where this mean exceeds `mean_global + n_sigma * std_global` are outliers.
This is the standard preprocessing step in Open3D and PCL.

In [ ]:
def statistical_outlier_removal(pts, k=20, n_sigma=2.0):
    kdt   = KDTree(pts)
    dists, _ = kdt.query(pts, k=k+1)  # +1 because first result is self
    mean_dists = dists[:, 1:].mean(axis=1)
    threshold  = mean_dists.mean() + n_sigma * mean_dists.std()
    mask = mean_dists <= threshold
    return pts[mask], ~mask

cloud_clean, outlier_mask = statistical_outlier_removal(cloud, k=20, n_sigma=2.0)
print(f'Original:  {len(cloud):,} points')
print(f'Cleaned:   {len(cloud_clean):,} points')
print(f'Removed:   {outlier_mask.sum()} outliers')

fig = plt.figure(figsize=(13, 5))
for i, (pts, title, c) in enumerate([
    (cloud,       'Before SOR', '#8A857E'),
    (cloud_clean, 'After SOR (outliers removed)', 'steelblue')]):
    ax = fig.add_subplot(1, 2, i+1, projection='3d')
    ax.scatter(pts[:,0], pts[:,1], pts[:,2], s=1, c=c, alpha=0.4)
    if i == 1:
        outliers = cloud[outlier_mask]
        ax.scatter(outliers[:,0], outliers[:,1], outliers[:,2], s=15, c='tomato', label='outliers')
        ax.legend(markerscale=2, fontsize=8)
    ax.set_title(title, fontsize=9)
plt.tight_layout(); plt.show()

---
## Part 4 -- RANSAC Plane Segmentation

RANSAC (Random Sample Consensus) robustly fits a model despite outliers:
1. Sample 3 random points, fit a plane
2. Count inliers (points within threshold distance)
3. Repeat N times, keep the best model

This is how AV pipelines segment the ground plane from LiDAR scans.

In [ ]:
def fit_plane_from_3pts(p1, p2, p3):
    """Fit plane ax+by+cz+d=0 from 3 points. Returns (normal, d)."""
    v1 = p2 - p1; v2 = p3 - p1
    n = np.cross(v1, v2)
    norm = np.linalg.norm(n)
    if norm < 1e-10: return None, None
    n = n / norm
    return n, -np.dot(n, p1)

def ransac_plane(pts, n_iters=200, dist_thresh=0.05):
    best_inliers = np.array([], dtype=int)
    for _ in range(n_iters):
        idx = np.random.choice(len(pts), 3, replace=False)
        n, d = fit_plane_from_3pts(pts[idx[0]], pts[idx[1]], pts[idx[2]])
        if n is None: continue
        dists    = np.abs(pts @ n + d)
        inliers  = np.where(dists < dist_thresh)[0]
        if len(inliers) > len(best_inliers):
            best_inliers = inliers
    # Refit on all inliers
    if len(best_inliers) >= 3:
        A = np.column_stack([pts[best_inliers,:2], np.ones(len(best_inliers))])
        coef, _, _, _ = np.linalg.lstsq(A, pts[best_inliers,2], rcond=None)
        return best_inliers, coef
    return best_inliers, None

ground_mask_idx, coef = ransac_plane(cloud_clean, n_iters=300, dist_thresh=0.06)
ground = cloud_clean[ground_mask_idx]
remaining = np.delete(cloud_clean, ground_mask_idx, axis=0)
print(f'Ground plane: {len(ground):,} inliers  |  remaining: {len(remaining):,} pts')

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(ground[:,0],    ground[:,1],    ground[:,2],    s=1, c='#8A857E', alpha=0.3, label='Ground plane')
ax.scatter(remaining[:,0], remaining[:,1], remaining[:,2], s=2, c='steelblue', alpha=0.6, label='Objects')
ax.set_title(f'RANSAC plane segmentation ({len(ground):,} ground pts)')
ax.legend(markerscale=5, fontsize=8)
plt.tight_layout(); plt.show()

---
## Exercises

### Exercise 1 -- Radius search
Implement `radius_search(pts, query_pt, radius)` using the KD-tree. Compare to k-NN: which is better for normal estimation?

### Exercise 2 -- Euclidean clustering
After ground removal, cluster the remaining points by proximity (DBSCAN or region growing). Each cluster = one object.

### Exercise 3 -- Depth image projection
Project the 3D point cloud onto a 2D image by binning points into a (row, col) grid. The pixel value = nearest z in that bin.

---
## Summary: Production Equivalents

| Operation | This notebook | Open3D | PCL |
|-----------|--------------|--------|-----|
| Voxel downsample | `voxel_downsample()` | `o3d.geometry.PointCloud.voxel_down_sample()` | `VoxelGrid` filter |
| Normal estimation | `estimate_normals()` | `estimate_normals()` | `NormalEstimation` |
| SOR filter | `statistical_outlier_removal()` | `remove_statistical_outlier()` | `StatisticalOutlierRemoval` |
| RANSAC plane | `ransac_plane()` | `segment_plane()` | `SACSegmentation` |
| KD-tree | `scipy.spatial.KDTree` | `o3d.geometry.KDTreeFlann` | `pcl::KdTree` |

In [ ]:
print('Notebook complete.')
print('Key takeaways:')
print('  Voxel grid  -> uniform spatial resolution, fast')
print('  PCA normals -> local neighborhood plane fitting')
print('  SOR filter  -> remove statistical outliers')
print('  RANSAC      -> robust model fitting with outliers')
print()
print('Next: ICP registration (SLAM NB02) and surface reconstruction (this track NB02)')